## ▶ What you'll see when you run this
- A table of short strings with their **token counts** printed instantly — no API key needed for the first result.

**Time:** ~10 min · **Cost:** free (defaults to cheapest model: Gemini Flash / Claude Haiku / local Ollama) · **Keys:** none for the first cell; **ANTHROPIC_API_KEY** for the live-model cells

# Week 8 · Notebook 1 — How LLMs Work
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Hands-on tour of the core ideas: **tokens**, the **context window**, **sampling / temperature**, **in-context learning**, and **hallucination**.

> Some cells call the **Claude** API. In Colab use the 🔑 *Secrets* panel and `userdata.get('ANTHROPIC_API_KEY')`; locally use an environment variable. **Never commit keys.**

## 0. Wow in 5 seconds — count tokens with no API key
Pure Python, runs immediately. Each emoji and rare word costs **more than one token**.

In [ ]:
# A crude byte-based token estimate so this cell ALWAYS runs (no install, no key).
def rough_tokens(text):
    # ~4 bytes per token is the classic rule of thumb for English.
    return max(1, round(len(text.encode('utf-8')) / 4))

for s in ['AI', 'Artificial intelligence', 'strawberry', '🍓🍓🍓']:
    print(f'{rough_tokens(s):2d} est. tokens | {s!r}')

## 0b. Install (for the exact, model-grade tokenizer + live calls)

In [ ]:
!pip -q install anthropic transformers

## 1. Tokens & tokenization
Models see **tokens** (sub-word IDs), not characters. We use a public BPE tokenizer to *illustrate* the idea; exact counts differ per model.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

samples = [
    'AI',
    'Artificial intelligence is reshaping software.',
    'antidisestablishmentarianism',
]
for s in samples:
    ids = tok.encode(s)
    pieces = [tok.decode([i]) for i in ids]
    print(f'{len(ids):2d} tokens | {s!r}')
    print('     pieces:', pieces)


**Rule of thumb:** 1 token ≈ 4 characters ≈ 0.75 words. Notice the rare word splits into several tokens.

**Task (A7):** add three strings of your own above and predict the token count before you run it.

## 2. Exact token counting with Claude
Anthropic exposes an exact token counter so you can budget prompts.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass

import anthropic
client = anthropic.Anthropic()
try:
    ct = client.messages.count_tokens(
        model='claude-sonnet-4-6',
        messages=[{'role': 'user', 'content': 'Explain tokens in one sentence.'}],
    )
    print('Claude input tokens:', ct.input_tokens)
except Exception as e:
    print('Set ANTHROPIC_API_KEY to run this cell:', e)

## 3. The context window
Prompt **plus** reply must fit the model's context window. There is **no memory** between calls — a chat app works by *replaying* the whole conversation each turn.

In [ ]:
# A 'conversation' is just a growing list of messages you resend every turn.
conversation = [
    {'role': 'user', 'content': 'My favorite number is 7.'},
    {'role': 'assistant', 'content': 'Got it — your favorite number is 7.'},
    {'role': 'user', 'content': 'What is my favorite number?'},
]
try:
    msg = client.messages.create(model='claude-sonnet-4-6', max_tokens=50,
                                 messages=conversation)
    print(msg.content[0].text)
    print('--- if we DROP the history, the model can no longer know: ---')
    msg2 = client.messages.create(model='claude-sonnet-4-6', max_tokens=50,
        messages=[{'role':'user','content':'What is my favorite number?'}])
    print(msg2.content[0].text)
except Exception as e:
    print('Set ANTHROPIC_API_KEY to run this cell:', e)

## 4. Sampling & temperature
Run the **same prompt** at `temperature=0.0` (focused) and `temperature=1.0` (diverse) a few times and compare.

In [ ]:
def ask(prompt, temperature, n=3):
    outs = []
    for _ in range(n):
        m = client.messages.create(model='claude-sonnet-4-6', max_tokens=40,
                                   temperature=temperature,
                                   messages=[{'role':'user','content':prompt}])
        outs.append(m.content[0].text.strip().replace('\n',' '))
    return outs

P = 'Give a one-line tagline for a coffee shop run by robots.'
try:
    print('TEMP 0.0 (deterministic):')
    for o in ask(P, 0.0): print('  -', o)
    print('TEMP 1.0 (creative):')
    for o in ask(P, 1.0): print('  -', o)
except Exception as e:
    print('Set ANTHROPIC_API_KEY to run this cell:', e)

**Task (A7):** describe in 2–3 sentences how the temp-0 and temp-1 outputs differed. When would you want each?

## 5. In-context learning (zero-shot vs few-shot)
Few-shot examples teach the task **and** lock the output format — no retraining.

In [ ]:
few_shot = (
    'Classify the review sentiment as positive or negative.\n'
    'Review: "Loved every minute." -> positive\n'
    'Review: "A complete waste of time." -> negative\n'
    'Review: "The acting saved an otherwise dull script." ->'
)
try:
    m = client.messages.create(model='claude-sonnet-4-6', max_tokens=10,
        temperature=0, messages=[{'role':'user','content':few_shot}])
    print('Few-shot answer:', m.content[0].text.strip())
except Exception as e:
    print('Set ANTHROPIC_API_KEY to run this cell:', e)

## 6. Hallucination
LLMs generate *plausible* text, not verified fact. Ask about something that does not exist and watch a confident, wrong answer appear.

In [ ]:
trap = ('Summarize the famous 2019 paper "Quantum Transformers for '
        'Underwater Basket Weaving" by Dr. Eleanor Fakename.')
try:
    m = client.messages.create(model='claude-sonnet-4-6', max_tokens=150,
        temperature=0, messages=[{'role':'user','content':trap}])
    print(m.content[0].text)
except Exception as e:
    print('Set ANTHROPIC_API_KEY to run this cell:', e)

**Task (A7):** craft your **own** trap prompt that yields a confident but false answer. Paste the output and write 2–3 sentences on *why* it happened and how you would mitigate it (temperature=0, ask for sources, RAG, tools, verify).

---
### Recap
Tokens → context window → sampling → in-context learning → hallucination. Next notebook: call **Claude**, **Gemini**, and a **local** model and compare them.

### 🎯 Start My Assistant — *just pick* (2 minutes)
Your **final project** begins this week, but the only thing to do now is **pick your idea and track** — RAG / Tool-Using Agent / Multimodal / Fine-Tuned (see `Final-Project-Capstone.md`). **Don't build the app this week.** The graded proposal + a 'wow in 5 min' prototype is **Milestone M1, due Week 9** (`capstone/M1.md`). Jot one line below — that's the whole task.

In [ ]:
my_assistant_idea  = ''   # <- one line: what should your assistant help with?
my_assistant_track = ''   # <- one of: RAG | Agent | Multimodal | Fine-Tuned
print('Idea :', my_assistant_idea or '(fill me in — building comes in Week 9)')
print('Track:', my_assistant_track or '(pick one)')